# Bank Credit Card Fraud Detection - Exploratory Data Analysis & Preprocessing
This notebook loads the anonymized PCA bank transaction data, cleans it, performs exploratory data analysis, scales features, and saves the cleaned dataset for modeling.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Append src path
sys.path.append('..')
from src.data_preprocessing import load_data, clean_creditcard_data, scale_and_encode


## 1. Load Dataset
We load the `creditcard.csv` file using our modular load function.


In [ ]:
_, _, cc_df = load_data('../data')
print("Raw Credit Card Data Shape:", cc_df.shape)


## 2. Data Cleaning
We search for and drop duplicates from the credit card dataset to make sure the model is trained on distinct records.


In [ ]:
print("Duplicates found:", cc_df.duplicated().sum())
cleaned_cc = clean_creditcard_data(cc_df)
print("Cleaned Dataset Shape (duplicates removed):", cleaned_cc.shape)
print("Missing values:", cleaned_cc.isnull().sum().sum())


## 3. Exploratory Data Analysis
Let's analyze the extreme class imbalance and distributions of the PCA dataset's non-anonymized features: `Time` and `Amount`.


In [ ]:
class_counts = cleaned_cc["Class"].value_counts()
class_pct = cleaned_cc["Class"].value_counts(normalize=True) * 100
print("Class Distribution:")
print(class_counts)
print("\nPercentage:")
print(class_pct)

plt.figure(figsize=(5,4))
sns.countplot(data=cleaned_cc, x="Class", palette='Set1')
plt.title("Credit Card Fraud Class Distribution")
plt.xlabel("Class (0 = Legit, 1 = Fraud)")
plt.ylabel("Count (log-scale)")
plt.yscale("log")
plt.show()


In [ ]:
# Amount distribution for legit and fraud transactions
plt.figure(figsize=(8,5))
sns.histplot(cleaned_cc[cleaned_cc["Class"] == 1]["Amount"], bins=30, color='red', label='Fraud', kde=True)
sns.histplot(cleaned_cc[cleaned_cc["Class"] == 0]["Amount"], bins=30, color='blue', label='Legit', kde=True, alpha=0.3)
plt.title("Transaction Amount Distribution by Class")
plt.xlabel("Amount ($)")
plt.ylabel("Count")
plt.xlim(0, 2000)
plt.legend()
plt.show()


In [ ]:
# Time distribution density
plt.figure(figsize=(8,5))
sns.kdeplot(data=cleaned_cc, x="Time", hue="Class", common_norm=False, fill=True, palette='Set2')
plt.title("Transaction Time Density by Class")
plt.xlabel("Time (Seconds elapsed)")
plt.ylabel("Density")
plt.show()


## 4. Scaling and Preprocessing
The features `V1-V28` are already PCA-transformed (centered and scaled). We scale the raw `Amount` and `Time` columns using `StandardScaler` to align them with the PCA features, and save the dataset.


In [ ]:
_, scaled_cc = scale_and_encode(None, cleaned_cc)
print("Processed Dataset Shape:", scaled_cc.shape)

# Save processed data
os.makedirs('../data/processed', exist_ok=True)
scaled_cc.to_csv('../data/processed/processed_creditcard.csv', index=False)
print("Successfully saved processed credit card dataset to ../data/processed/processed_creditcard.csv")
